# 16 — ML V2: Multi-Task Daily Ranking

Bu son kontrollü ML araştırmasıdır. Amaç, Baseline Robot'un giriş ve çıkış
kurallarını değiştirmeden günlük AL adaylarının sırasını iyileştirmektir.

Üç ayrı hedef kullanılır:

1. `Big_Winner_Label_2R`: İşlem en az 2R kazandırır mı?
2. `Expected_R_Target`: Beklenen kırpılmış R Multiple
3. `Stop_Risk_Label`: Stop veya büyük kayıp olasılığı

Model çıktıları günlük çapraz kesit sıralarına çevrilir ve orijinal Robot
sırasıyla birleştirilir.

Metodoloji:

- Development 2018–2022: görev bazlı model seçimi
- Validation 2023–2024: ağırlık ve Top-K politika seçimi
- 2025+: yalnızca Validation'da kabul edilen tek konfigürasyonla aylık
  expanding walk-forward audit
- CAGR üstünlük şartı: en az +3 yüzde puan
- Eşik, ağırlık veya model Audit sonucuna bakılarak değiştirilmez


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = (
    Path.cwd()
    if (Path.cwd() / "src").exists()
    else Path.cwd().parent
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

from src.features import add_indicators
from src.signals import (
    build_market_regime,
    add_robot_scores,
)
from src.presets import (
    FINAL_STRATEGY_CONFIG,
    FINAL_PORTFOLIO_CONFIG,
)
from src.ml_dataset import (
    add_meta_features,
)
from src.ml_training import (
    create_purged_expanding_folds,
)
from src.ml_v2 import (
    V2_FEATURE_COLUMNS,
    DEFAULT_WEIGHT_PROFILES,
    DEFAULT_POLICIES,
    V2WalkForwardConfig,
    add_v2_features,
    build_v2_event_dataset,
    build_v2_classifier_models,
    build_v2_regressor_models,
    evaluate_classifier_candidates,
    summarize_classifier_cv,
    evaluate_regressor_candidates,
    summarize_regressor_cv,
    select_v2_models,
    fit_v2_models,
    score_v2_components,
    run_v2_validation_grid,
    v2_acceptance_table,
    select_v2_validation_champion,
    generate_v2_walk_forward_components,
    evaluate_v2_walk_forward,
    save_v2_artifacts,
)


## 1. Tam geçmiş fiyatları ve olay veri setini yükle


In [2]:
stock_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "bist100_robot_clean.parquet"
)

market_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "xu100_robot_clean.parquet"
)

events = pd.read_parquet(
    PROJECT_ROOT
    / "results"
    / "ml"
    / "robot_meta_label_training.parquet"
)

for column in [
    "Signal_Date",
    "Entry_Date",
    "Exit_Date",
]:
    events[column] = pd.to_datetime(
        events[column]
    )

print(
    "Hisse dönemi:",
    stock_prices["Date"].min(),
    "→",
    stock_prices["Date"].max(),
)
print(
    "XU100 dönemi:",
    market_prices["Date"].min(),
    "→",
    market_prices["Date"].max(),
)
print("Olay sayısı:", len(events))


Hisse dönemi: 2018-01-02 00:00:00 → 2026-07-27 00:00:00
XU100 dönemi: 2018-01-02 00:00:00 → 2026-07-23 00:00:00
Olay sayısı: 3879


## 2. Robot, mevcut ML ve yeni breadth özelliklerini oluştur


In [3]:
stock_features = add_indicators(stock_prices)
market_features = add_indicators(market_prices)
market_regime = build_market_regime(
    market_features
)

scored_prices = add_robot_scores(
    stock_features=stock_features,
    market_regime=market_regime,
    config=FINAL_STRATEGY_CONFIG,
    include_reasons=False,
)

featured_prices = add_meta_features(
    scored_prices=scored_prices,
    market_features=market_features,
)

v2_prices = add_v2_features(
    featured_prices
)

v2_events = build_v2_event_dataset(
    events=events,
    v2_prices=v2_prices,
)

target_summary = (
    v2_events.groupby("Period")
    .agg(
        Event_Count=("R_Multiple", "size"),
        Big_Winner_Rate=(
            "Big_Winner_Label_2R",
            "mean",
        ),
        Stop_Risk_Rate=(
            "Stop_Risk_Label",
            "mean",
        ),
        Average_R=(
            "R_Multiple",
            "mean",
        ),
        Median_R=(
            "R_Multiple",
            "median",
        ),
    )
    .reset_index()
)

target_summary["Big_Winner_Rate_%"] = (
    target_summary["Big_Winner_Rate"] * 100
)
target_summary["Stop_Risk_Rate_%"] = (
    target_summary["Stop_Risk_Rate"] * 100
)

display(target_summary)
print("ML V2 özellik sayısı:", len(V2_FEATURE_COLUMNS))


,Period,Event_Count,Big_Winner_Rate,Stop_Risk_Rate,Average_R,Median_R,Big_Winner_Rate_%,Stop_Risk_Rate_%
0,Audit_2025_Plus,930,0.135484,0.624731,0.210730,-1.000000,13.548387,62.473118
1,Development,1851,0.235008,0.483522,1.140254,-0.639523,23.500810,48.352242
2,Validation,1098,0.173953,0.529144,0.627123,-0.894113,17.395264,52.914390


ML V2 özellik sayısı: 48


## 3. Development purged CV fold'ları


In [4]:
development = (
    v2_events.loc[
        v2_events["Period"].eq("Development")
    ]
    .sort_values(["Signal_Date", "Ticker"])
    .reset_index(drop=True)
)

validation = (
    v2_events.loc[
        v2_events["Period"].eq("Validation")
    ]
    .sort_values(["Signal_Date", "Ticker"])
    .reset_index(drop=True)
)

audit = (
    v2_events.loc[
        v2_events["Period"].eq("Audit_2025_Plus")
    ]
    .sort_values(["Signal_Date", "Ticker"])
    .reset_index(drop=True)
)

folds = create_purged_expanding_folds(
    development_data=development,
    n_splits=4,
    initial_train_fraction=0.40,
    embargo_days=5,
)

fold_rows = []
for fold in folds:
    fold_rows.append(
        {
            "Fold": fold.fold,
            "Train_Count": len(
                fold.train_indices
            ),
            "Validation_Count": len(
                fold.validation_indices
            ),
            "Training_Exit_End": (
                fold.train_exit_end
            ),
            "Validation_Start": (
                fold.validation_start
            ),
            "Purged_Correctly": (
                fold.train_exit_end
                < fold.validation_start
            ),
        }
    )

fold_table = pd.DataFrame(fold_rows)
display(fold_table)

if not fold_table[
    "Purged_Correctly"
].all():
    raise RuntimeError(
        "Purged fold kontrolü başarısız."
    )


,Fold,Train_Count,Validation_Count,Training_Exit_End,Validation_Start,Purged_Correctly
0,1,671,285,2020-10-08,2020-10-19,True
1,2,1006,239,2021-04-21,2021-04-27,True
2,3,1192,284,2021-12-03,2021-12-13,True
3,4,1501,314,2022-06-17,2022-06-24,True


## 4. İki sınıflandırma görevi için model seçimi


In [5]:
classifier_models = (
    build_v2_classifier_models(
        V2_FEATURE_COLUMNS,
        random_state=42,
    )
)

classifier_fold_frames = []
classifier_prediction_frames = []

for target_column in [
    "Big_Winner_Label_2R",
    "Stop_Risk_Label",
]:
    fold_metrics, predictions = (
        evaluate_classifier_candidates(
            development_data=development,
            folds=folds,
            feature_columns=V2_FEATURE_COLUMNS,
            target_column=target_column,
            models=classifier_models,
        )
    )
    classifier_fold_frames.append(
        fold_metrics
    )
    classifier_prediction_frames.append(
        predictions
    )

classifier_fold_metrics = pd.concat(
    classifier_fold_frames,
    ignore_index=True,
)

classifier_cv_summary = (
    summarize_classifier_cv(
        classifier_fold_metrics
    )
)

display(classifier_cv_summary)


,Task,Model,PR_AUC_Mean,PR_AUC_Min,PR_AUC_Lift_Mean,PR_AUC_Lift_Min,ROC_AUC_Mean,ROC_AUC_Min,Brier_Mean,Log_Loss_Mean,Fold_Count
0,Big_Winner_Label_2R,LogisticRegression,0.342524,0.230183,0.082420,0.052410,0.638728,0.566964,0.194041,0.574007,4
1,Big_Winner_Label_2R,ExtraTreesClassifier,0.341616,0.194298,0.081511,0.043420,0.606458,0.559699,0.204395,0.598121,4
2,Big_Winner_Label_2R,HistGradientBoostingClassifier,0.296464,0.165258,0.036359,0.013500,0.558170,0.537767,0.236145,0.881055,4
3,Stop_Risk_Label,LogisticRegression,0.548640,0.421724,0.073837,-0.000871,0.549408,0.454943,0.333035,0.978184,4
4,Stop_Risk_Label,ExtraTreesClassifier,0.514255,0.357081,0.039453,-0.012345,0.526194,0.494471,0.261242,0.716949,4
5,Stop_Risk_Label,HistGradientBoostingClassifier,0.481465,0.350009,0.006663,-0.020048,0.494857,0.420373,0.344261,0.985930,4


## 5. Expected R regresyon modeli seçimi


In [6]:
regressor_models = (
    build_v2_regressor_models(
        V2_FEATURE_COLUMNS,
        random_state=42,
    )
)

(
    regressor_fold_metrics,
    regressor_predictions,
) = evaluate_regressor_candidates(
    development_data=development,
    folds=folds,
    feature_columns=V2_FEATURE_COLUMNS,
    target_column="Expected_R_Target",
    models=regressor_models,
)

regressor_cv_summary = (
    summarize_regressor_cv(
        regressor_fold_metrics
    )
)

display(regressor_cv_summary)

model_selection = select_v2_models(
    classifier_summary=classifier_cv_summary,
    regressor_summary=regressor_cv_summary,
)

print("ML V2 MODEL SEÇİMİ")
print(model_selection)


,Task,Model,Spearman_Mean,Spearman_Min,Top20_R_Lift_Mean,Top20_R_Lift_Min,MAE_Mean,RMSE_Mean,Fold_Count
0,Expected_R_Target,ExtraTreesRegressor,0.129522,0.090654,0.436784,0.066716,1.770400,2.262199,4
1,Expected_R_Target,HistGradientBoostingRegressor,0.119010,-0.016858,0.253829,-0.167265,1.783436,2.386455,4
2,Expected_R_Target,RandomForestRegressor,0.098779,0.034264,0.186249,-0.018573,1.784581,2.277334,4
3,Expected_R_Target,HuberRegressor,0.037746,-0.190085,0.053652,-0.725565,2.223689,3.026522,4


ML V2 MODEL SEÇİMİ
V2ModelSelection(big_winner_model='LogisticRegression', expected_r_model='ExtraTreesRegressor', stop_risk_model='LogisticRegression')


Model seçiminde yalnızca Development purged CV kullanıldı. Validation
portföy performansı model tipini değiştirmek için kullanılmayacak.


## 6. Development'ta eğit ve Validation günlük AL satırlarını skorla


In [7]:
VALIDATION_START = "2023-01-01"
VALIDATION_END = "2024-12-31"
validation_purge_boundary = (
    pd.Timestamp(VALIDATION_START)
    - pd.Timedelta(days=5)
)

eligible_development = development.loc[
    development["Exit_Date"]
    < validation_purge_boundary
].copy()

validation_models = fit_v2_models(
    training_data=eligible_development,
    feature_columns=V2_FEATURE_COLUMNS,
    selection=model_selection,
    random_state=42,
)

validation_mask = (
    v2_prices["Date"].between(
        VALIDATION_START,
        VALIDATION_END,
        inclusive="both",
    )
    & v2_prices["Signal"].eq("AL")
)

validation_component_prices = (
    v2_prices.copy()
)

scored_validation_rows = score_v2_components(
    rows=v2_prices.loc[
        validation_mask
    ].copy(),
    models=validation_models,
    feature_columns=V2_FEATURE_COLUMNS,
)

for column in [
    "MLV2_P_BIG_WINNER",
    "MLV2_EXPECTED_R",
    "MLV2_P_STOP",
]:
    validation_component_prices[column] = np.nan
    validation_component_prices.loc[
        validation_mask,
        column,
    ] = scored_validation_rows[
        column
    ].to_numpy()

print(
    "Validation Robot AL:",
    int(validation_mask.sum()),
)
display(
    scored_validation_rows[
        [
            "Date",
            "Ticker",
            "Score",
            "MLV2_P_BIG_WINNER",
            "MLV2_EXPECTED_R",
            "MLV2_P_STOP",
        ]
    ].describe()
)


Validation Robot AL: 7534


,Date,Score,MLV2_P_BIG_WINNER,MLV2_EXPECTED_R,MLV2_P_STOP
count,7534,7534.000000,7534.000000,7534.000000,7534.000000
mean,2023-12-05 14:35:23.493000,12.289488,0.605475,0.589019,0.449630
min,2023-01-02 00:00:00,11.000000,0.000772,-0.233497,0.077109
25%,2023-07-10 00:00:00,12.000000,0.473722,0.411489,0.303303
50%,2023-11-21 12:00:00,12.000000,0.639092,0.592444,0.431978
75%,2024-05-02 00:00:00,13.000000,0.773078,0.788600,0.571409
max,2024-12-31 00:00:00,14.000000,0.972790,1.385357,0.992751
std,NaN,1.050663,0.214478,0.292407,0.187689


## 7. Validation portföy grid'i


In [8]:
validation_grid, validation_outputs = (
    run_v2_validation_grid(
        component_prices=(
            validation_component_prices
        ),
        weight_profiles=(
            DEFAULT_WEIGHT_PROFILES
        ),
        policies=DEFAULT_POLICIES,
        strategy_config=(
            FINAL_STRATEGY_CONFIG
        ),
        portfolio_config=(
            FINAL_PORTFOLIO_CONFIG
        ),
        start=VALIDATION_START,
        end=VALIDATION_END,
    )
)

acceptance_table = v2_acceptance_table(
    validation_results=validation_grid,
    minimum_cagr_improvement_pp=3.0,
    minimum_trade_fraction=0.50,
    maximum_drawdown_deterioration_pp=2.0,
)

display(
    acceptance_table[
        [
            "Configuration",
            "Weight_Profile",
            "Policy",
            "CAGR_%",
            "CAGR_Difference_pp",
            "Max_Drawdown_%",
            "Profit_Factor",
            "Sharpe",
            "Calmar",
            "Trade_Count",
            "Signal_Pass_Rate_%",
            "Acceptance_Count",
            "MLV2_Accepted",
        ]
    ].head(20)
)

validation_champion, v2_accepted = (
    select_v2_validation_champion(
        acceptance_table
    )
)

print("ML V2 Validation kabul edildi mi?:", v2_accepted)
print("Validation champion:", validation_champion)


,Configuration,Weight_Profile,Policy,CAGR_%,CAGR_Difference_pp,Max_Drawdown_%,Profit_Factor,Sharpe,Calmar,Trade_Count,Signal_Pass_Rate_%,Acceptance_Count,MLV2_Accepted
0,Baseline_Robot,NaN,NaN,36.064257,0.000000,-23.126458,1.687332,1.575621,1.559437,214,100.000000,4,False
1,ML_Only__Top4_UpperHalf,ML_Only,Top4_UpperHalf,35.014693,-1.049564,-21.729321,1.717911,1.633770,1.611403,202,23.400584,4,False
2,ML_Only__Top4,ML_Only,Top4,34.710142,-1.354115,-21.687846,1.675133,1.619016,1.600442,207,24.369525,3,False
3,ML_Only__Top6,ML_Only,Top6,34.332910,-1.731347,-22.137525,1.665564,1.603956,1.550892,208,34.828776,3,False
4,ML_Only__Top3,ML_Only,Top3,32.814032,-3.250225,-22.480202,1.628252,1.540894,1.459686,202,18.622246,2,False
5,Trend_Focus__Top3,Trend_Focus,Top3,29.969040,-6.095217,-20.410760,1.622376,1.404006,1.468296,205,18.622246,2,False
6,ML_Only__Top6_UpperHalf,ML_Only,Top6_UpperHalf,28.129810,-7.934447,-22.344048,1.604105,1.371482,1.258940,207,31.935227,2,False
7,Balanced__Top6,Balanced,Top6,27.327828,-8.736429,-24.891969,1.548511,1.304134,1.097857,210,34.828776,2,False
8,Trend_Focus__Top6_UpperHalf,Trend_Focus,Top6_UpperHalf,26.179249,-9.885008,-21.668609,1.511610,1.250925,1.208165,212,31.961773,2,False
9,Trend_Focus__Top4_UpperHalf,Trend_Focus,Top4_UpperHalf,26.028352,-10.035905,-21.833748,1.509611,1.244686,1.192116,212,23.400584,2,False


ML V2 Validation kabul edildi mi?: False
Validation champion: None


Kabul şartlarının tamamı:

- Baseline'a göre en az +3 CAGR puanı
- Profit Factor düşmeyecek
- Sharpe düşmeyecek
- Max drawdown en fazla 2 puan kötüleşecek
- Baseline işlem sayısının en az %50'si korunacak

Hiçbir aday geçmezse ML V2 araştırması burada başarısız sayılır ve
2025+ sonuçlarına bakılarak yeni bir konfigürasyon seçilmez.


## 8. Kabul edilen tek konfigürasyonla 2025+ walk-forward audit


In [9]:
walk_forward_metrics = None
walk_forward_equity = None
walk_forward_yearly = None
walk_forward_active = None
walk_forward_log = None
walk_forward_outputs = None

if v2_accepted:
    selected_profile = next(
        profile
        for profile in DEFAULT_WEIGHT_PROFILES
        if profile.name
        == validation_champion[
            "Weight_Profile"
        ]
    )

    selected_policy = next(
        policy
        for policy in DEFAULT_POLICIES
        if policy.name
        == validation_champion["Policy"]
    )

    WALK_FORWARD_START = "2025-01-01"
    WALK_FORWARD_END = min(
        v2_prices["Date"].max(),
        market_prices["Date"].max(),
    ).strftime("%Y-%m-%d")

    walk_forward_component_prices, (
        walk_forward_log
    ) = generate_v2_walk_forward_components(
        v2_prices=v2_prices,
        event_data=v2_events,
        selection=model_selection,
        feature_columns=V2_FEATURE_COLUMNS,
        config=V2WalkForwardConfig(
            start=WALK_FORWARD_START,
            end=WALK_FORWARD_END,
            retrain_frequency="MS",
            embargo_days=5,
            minimum_training_events=500,
            random_state=42,
        ),
    )

    (
        walk_forward_metrics,
        walk_forward_equity,
        walk_forward_yearly,
        walk_forward_active,
        walk_forward_outputs,
    ) = evaluate_v2_walk_forward(
        component_prices=(
            walk_forward_component_prices
        ),
        profile=selected_profile,
        policy=selected_policy,
        market_prices=market_prices,
        strategy_config=(
            FINAL_STRATEGY_CONFIG
        ),
        portfolio_config=(
            FINAL_PORTFOLIO_CONFIG
        ),
        start=WALK_FORWARD_START,
        end=WALK_FORWARD_END,
    )

    print("WALK-FORWARD PERFORMANS")
    display(walk_forward_metrics)
    display(walk_forward_active)
    display(walk_forward_yearly)

    print("WALK-FORWARD EĞİTİM DENETİMİ")
    display(walk_forward_log)

else:
    print(
        "Validation kabul şartları sağlanmadığı için "
        "2025+ audit çalıştırılmadı."
    )


Validation kabul şartları sağlanmadığı için 2025+ audit çalıştırılmadı.


## 9. Walk-forward grafik


In [10]:
if walk_forward_equity is not None:
    chart = walk_forward_equity.set_index(
        "Date"
    )

    plt.figure(figsize=(13, 7))
    plt.plot(
        chart.index,
        chart["Baseline_Robot"],
        label="Baseline Robot",
    )
    plt.plot(
        chart.index,
        chart["ML_V2_Challenger"],
        label="ML V2 Challenger",
    )
    plt.plot(
        chart.index,
        chart["BIST100_Gross"],
        label="BIST100 Gross",
    )
    plt.title(
        "ML V2 Walk-Forward — "
        "Baseline, Challenger ve BIST100"
    )
    plt.xlabel("Tarih")
    plt.ylabel("Portföy Değeri (TL)")
    plt.legend()
    plt.tight_layout()
    plt.show()


## 10. Sonuçları kaydet


In [11]:
OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "ml"
    / "v2"
)

artifact_paths = save_v2_artifacts(
    output_directory=OUTPUT_DIR,
    classifier_cv=classifier_cv_summary,
    regressor_cv=regressor_cv_summary,
    model_selection=model_selection,
    validation_grid=validation_grid,
    acceptance_table=acceptance_table,
    champion=validation_champion,
    walk_forward_metrics=(
        walk_forward_metrics
    ),
    walk_forward_equity=(
        walk_forward_equity
    ),
    walk_forward_yearly=(
        walk_forward_yearly
    ),
    walk_forward_active=(
        walk_forward_active
    ),
    walk_forward_log=(
        walk_forward_log
    ),
)

for name, path in artifact_paths.items():
    print(name, "→", path)


classifier_cv → c:\Users\okand\Desktop\Projects\Algorithmic Trading\BIST-Algo-Trade\results\ml\v2\v2_classifier_cv_summary.csv
regressor_cv → c:\Users\okand\Desktop\Projects\Algorithmic Trading\BIST-Algo-Trade\results\ml\v2\v2_regressor_cv_summary.csv
validation_grid → c:\Users\okand\Desktop\Projects\Algorithmic Trading\BIST-Algo-Trade\results\ml\v2\v2_validation_portfolio_grid.csv
acceptance → c:\Users\okand\Desktop\Projects\Algorithmic Trading\BIST-Algo-Trade\results\ml\v2\v2_validation_acceptance.csv
decision → c:\Users\okand\Desktop\Projects\Algorithmic Trading\BIST-Algo-Trade\results\ml\v2\v2_decision.json


## Karar ilkesi

ML V2 şu üç sonuçtan biriyle kapanır:

1. **Validation başarısız:** Baseline Robot korunur; V2 araştırması durur.
2. **Validation başarılı, walk-forward zayıf:** V2 canlı/paper sisteme alınmaz.
3. **Validation ve walk-forward güçlü:** V2 yalnızca yeni bir challenger olarak
   dual paper trading'e eklenir; Baseline hemen değiştirilmez.

Bu deneyden sonra yeni model/weight/policy taraması yapmamak gerekir. Sürekli
model denemek 2025+ dönemine dolaylı overfitting oluşturur.
